In [4]:
import numpy as np
import scipy.sparse.linalg as spla

from core.config import HelmholtzConfig
from core.resolution import grid_from_ppw_with_pml_extension
from core.medium import build_medium
from core.rhs import make_rhs  # or whatever you use

from operators.assemble import assemble_helmholtz_matrix

from diagnostics import plot_solution_with_pml, plot_sigma_map, plot_pml_bounds


ImportError: cannot import name 'make_rhs' from 'core.rhs' (C:\Users\31624\Documents\MIT\Programming\Freq2Transfer\src\core\rhs.py)

In [2]:
OMEGA_HIGH_LIST = [32.0, 64.0, 128.0]
def omega_low_from_high(omega_high: float) -> float:
    return omega_high / 2.0


In [ ]:
PML_SWEEP = dict(
    npml_list=[20, 30, 40, 50],
    eta_list=[3.0, 4.5, 6.0, 7.5],
    p=2,  # or whatever Laurent used as polynomial order
)


In [ ]:
for omega_high in OMEGA_HIGH_LIST:
    cfg = HelmholtzConfig(
        omega=omega_high,
        ppw=10.0,
        lx=1.0, ly=1.0,
        c_min=1.0,
    )

    grid = grid_from_ppw_with_pml_extension(
        omega=cfg.omega,
        ppw=cfg.ppw,
        lx=cfg.lx, ly=cfg.ly,
        c_min=cfg.c_min,
        # pass thickness npml here if your grid-extension function needs it
        npml=PML_DESIGN["npml"],
    )

    # choose_pml_config MUST implement Laurent:
    # Lpml = npml*h ; sigma_max = -(p+1)*log(R)/(2*Lpml)
    pml = choose_pml_config(grid=grid, omega=cfg.omega, **PML_DESIGN)

    sig = build_pml_profiles(grid, pml)

    print(f"\nomega={omega_high:>5.1f}  h={grid.h:.4e}  Lpml={pml.Lpml:.4e}  sigma_max={pml.sigma_max:.4e}")
    plot_pml_bounds(grid, pml)
    plot_sigma_map(grid, sig)


In [ ]:
def assert_sigma_ok(grid, sig):
    # adapt to your return type
    sx = sig["sigma_x"] if isinstance(sig, dict) else sig.sigma_x
    sy = sig["sigma_y"] if isinstance(sig, dict) else sig.sigma_y

    # You should have explicit index ranges for physical vs PML in your Grid2D.
    # Example names:
    i0, i1 = grid.phys_i0, grid.phys_i1
    j0, j1 = grid.phys_j0, grid.phys_j1

    assert np.allclose(sx[i0:i1+1], 0.0), "sigma_x nonzero in physical interior"
    assert np.allclose(sy[j0:j1+1], 0.0), "sigma_y nonzero in physical interior"


In [ ]:
def laurent_sigma_max(npml, h, p, R_target):
    Lpml = npml * h
    return -(p+1) * np.log(R_target) / (2 * Lpml)

# inside your omega loop, after pml is computed:
sigma_expected = laurent_sigma_max(PML_DESIGN["npml"], grid.h, PML_DESIGN["p"], PML_DESIGN["R_target"])
print("sigma_max expected / got:", sigma_expected, pml.sigma_max)
assert np.isclose(pml.sigma_max, sigma_expected, rtol=1e-12, atol=0), "sigma_max formula mismatch"
